In [5]:
from typing import List

from pyspark.sql import DataFrame, SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType

# Earth radius in miles
EARTH_RADIUS_MILES = 3958.8

# create a local Spark session
spark = SparkSession.builder \
    .master("local[*]") \
    .appName("pyspark-notebook") \
    .config("spark.sql.shuffle.partitions", "8") \
    .getOrCreate()

# alias for code that expects _spark
_spark = spark

spark

In [4]:
spark.stop()

In [ ]:
def word_tokenizer(dataframe):
    return (dataframe
        .withColumn("word", explode(split(dataframe.columns[0], r"\s+")))
        .withColumn("word", explode(split(dataframe.columns[0], "--")))
        .withColumn("word", lower(trim(regexp_replace("word", r"^[^\w]+|[^\w]+$", ""))))
        .select("word"))

In [ ]:
input_df = spark.read.text(input_path)


In [3]:
def compute_distance(_spark: SparkSession, dataframe: DataFrame) -> DataFrame:
    """
    Add a 'distance_miles' column to the dataframe using the Haversine formula.
    Uses Spark SQL functions (no Python UDF) for performance.
    Raises ValueError if required columns are missing.
    """
    required = _required_cols()
    missing = [c for c in required if c not in dataframe.columns]
    if missing:
        raise ValueError(f"Missing required columns for distance computation: {missing}")

    # Ensure numeric types
    df = dataframe
    for col in required:
        df = df.withColumn(col, F.col(col).cast(DoubleType()))

    # Convert degrees to radians
    lat1 = F.radians(F.col("start_station_latitude"))
    lon1 = F.radians(F.col("start_station_longitude"))
    lat2 = F.radians(F.col("end_station_latitude"))
    lon2 = F.radians(F.col("end_station_longitude"))

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = F.sin(dlat / 2) * F.sin(dlat / 2) + F.cos(lat1) * F.cos(lat2) * F.sin(dlon / 2) * F.sin(dlon / 2)
    c = 2 * F.atan2(F.sqrt(a), F.sqrt(1 - a))

    distance_miles = F.lit(EARTH_RADIUS_MILES) * c

    # Round to 2 decimals and handle null inputs by leaving distance null
    return df.withColumn("distance_miles", F.round(distance_miles, 2))

In [6]:
path = "/workspaces/dataengineer-transformations-python/resources/citibike/citibike.csv"
df_citibike = _spark.read.option("header", "true").option("inferSchema", "true").csv(path)
df_citibike.printSchema()
df_citibike.show(5, truncate=False)

root
 |-- tripduration: integer (nullable = true)
 |-- starttime: timestamp (nullable = true)
 |-- stoptime: timestamp (nullable = true)
 |-- start station id: integer (nullable = true)
 |-- start station name: string (nullable = true)
 |-- start station latitude: double (nullable = true)
 |-- start station longitude: double (nullable = true)
 |-- end station id: integer (nullable = true)
 |-- end station name: string (nullable = true)
 |-- end station latitude: double (nullable = true)
 |-- end station longitude: double (nullable = true)
 |-- bikeid: integer (nullable = true)
 |-- usertype: string (nullable = true)
 |-- birth year: string (nullable = true)
 |-- gender: integer (nullable = true)

+------------+-------------------+-------------------+----------------+------------------------------+----------------------+-----------------------+--------------+------------------------+--------------------+---------------------+------+----------+----------+------+
|tripduration|starttime  

In [8]:
def sanitize_columns(columns: List[str]) -> List[str]:
    return [column.replace(" ", "_") for column in columns]

df_citibike = df_citibike.toDF(*sanitize_columns(df_citibike.columns))
df_citibike.show(5, truncate=False)

+------------+-------------------+-------------------+----------------+------------------------------+----------------------+-----------------------+--------------+------------------------+--------------------+---------------------+------+----------+----------+------+
|tripduration|starttime          |stoptime           |start_station_id|start_station_name            |start_station_latitude|start_station_longitude|end_station_id|end_station_name        |end_station_latitude|end_station_longitude|bikeid|usertype  |birth_year|gender|
+------------+-------------------+-------------------+----------------+------------------------------+----------------------+-----------------------+--------------+------------------------+--------------------+---------------------+------+----------+----------+------+
|364         |2017-07-01 00:00:00|2017-07-01 00:06:05|539             |Metropolitan Ave & Bedford Ave|40.71534825           |-73.96024116           |3107          |Bedford Ave & Nassau Ave|40.7

In [10]:
new_df = compute_distance(_spark, df_citibike)
new_df.show(5, truncate=False)

Column<'radians(start_station_latitude)'>
+------------+-------------------+-------------------+----------------+------------------------------+----------------------+-----------------------+--------------+------------------------+--------------------+---------------------+------+----------+----------+------+--------------+
|tripduration|starttime          |stoptime           |start_station_id|start_station_name            |start_station_latitude|start_station_longitude|end_station_id|end_station_name        |end_station_latitude|end_station_longitude|bikeid|usertype  |birth_year|gender|distance_miles|
+------------+-------------------+-------------------+----------------+------------------------------+----------------------+-----------------------+--------------+------------------------+--------------------+---------------------+------+----------+----------+------+--------------+
|364         |2017-07-01 00:00:00|2017-07-01 00:06:05|539             |Metropolitan Ave & Bedford Ave|40.7

In [11]:
spark.stop()